# Phone test set: EXIF and conditions

Thin runner over `iiss.data.phone_exif` and `iiss.data.phone_metadata`. Evidence for assignment part 1 (own phone photos, at least 40 in total, at least 10 in hard conditions).

- EXIF is read from `dataset/manual-benchmark/originals/` (Roboflow strips it). GPS coordinates are never shown or written, only the `has_gps` flag.
- Writes `results/phone_exif.csv` and `results/phone_metadata_report.md` (same files as the CLIs). CPU only, a few seconds.
- `SMOKE = True` shows the same tables but writes nothing.

## Parameters

In [ ]:
SMOKE = False   # True: write no files

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from iiss.data.phone_exif import exif_summary, exif_table
from iiss.data.phone_metadata import (condition_exposure, condition_table, read_metadata,
                                      render_report, validate_metadata)
from iiss.utils.config import load_yaml
from iiss.utils.paths import repo_root, resolve_path

BASE = load_yaml(repo_root() / 'configs' / 'base.yaml')
RULES = BASE['phone_import']
PHONE = resolve_path(BASE['paths']['phone_root'])
RESULTS = resolve_path(BASE['paths']['results_root'])
pd.set_option('display.max_columns', 20, 'display.max_rows', 60)

## EXIF summary

In [ ]:
exif = exif_table(PHONE / RULES['originals_dir'])
if not SMOKE:
    exif.to_csv(RESULTS / 'phone_exif.csv', index=False)
exif_summary(exif).to_frame('value')

## EXIF per photo

In [ ]:
exif

## Metadata validation and condition counts

In [ ]:
meta = read_metadata(PHONE / RULES['metadata_file'])
report = validate_metadata(meta, exif, RULES)
text = render_report(report, RULES)
if not SMOKE:
    (RESULTS / 'phone_metadata_report.md').write_text(text, encoding='utf-8')
display(condition_table(report))
display(Markdown(text))

## Exposure per condition

Median (min-max) exposure time in seconds and ISO, and flash count, from EXIF, as a camera-side check of the `low_light` labels.

In [ ]:
condition_exposure(meta, exif)